In [0]:
CREATE VIEW workspace.gold.dim_customer AS
SELECT
	ROW_NUMBER() OVER (ORDER BY cst_id) AS customer_key,
	ci.cst_id AS customer_id,
	ci.cst_key AS customer_number,
	ci.cst_firstname AS first_name,
	ci.cst_lastname AS last_name,
	la.CNTRY AS country,
	ci.cst_marital_status AS marital_status,
	CASE WHEN ci.cst_gndr != 'n/a' THEN ci.cst_gndr
		ELSE COALESCE(ca.GEN, 'n/a')
	END AS gender,
	ca.BDATE AS birthdate,
	ci.cst_create_date AS create_date
FROM workspace.silver.cust_info_silver ci
LEFT JOIN workspace.silver.cust_az12_silver ca
ON ci.cst_key = ca.CID
LEFT JOIN workspace.silver.loc_a101_silver la
ON ci.cst_key = la.CID

In [0]:
CREATE VIEW workspace.gold.dim_products AS
SELECT 
ROW_NUMBER() OVER (ORDER BY pn.prd_start_dt, pn.prd_key) AS product_key,
	pn.prd_id AS product_id,
	pn.prd_key AS product_number,
	pn.prd_nm AS product_name,
	pn.cat_id AS category_id,
	pc.CAT AS category,
	pc.SUBCAT AS subcategory,
	pc.MAINTAINANCE AS maintainance,
	pn.prd_cost AS product_cost,
	pn.prd_line AS producy_line,
	pn.prd_start_dt AS startdate
FROM workspace.silver.prd_info_silver pn
LEFT JOIN workspace.silver.px_cat_g1v2_silver pc
ON pn.cat_id = pc.ID
WHERE prd_end_dt IS NULL

In [0]:
CREATE VIEW workspace.gold.fact_sales AS
SELECT
sd.sls_ord_num AS order_number,
pr.product_key,
cu.customer_key,
sd.sls_order_dt AS order_date,
sd.sls_ship_dt AS shipping_date,
sd.sls_due_dt AS due_date,
sd.sls_sales AS sales_amount,
sd.sls_quantity AS quantity,
sd.sls_price AS price
FROM workspace.silver.sales_details_silver sd
LEFT JOIN workspace.gold.dim_products pr
ON sd.sls_prd_key = pr.product_number
LEFT JOIN workspace.gold.dim_customer cu
ON sd.sls_cust_id = cu.customer_id

In [0]:
select * from workspace.gold.fact_sales;